In [9]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

In [10]:
data_dir = r"D:\pu\dataset\garbage_classification"

batch_size = 16
epochs = 5

In [11]:
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    data_dir,
    target_size=(224,224),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_gen = datagen.flow_from_directory(
    data_dir,
    target_size=(224,224),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

Found 5816 images belonging to 7 classes.
Found 1452 images belonging to 7 classes.


In [12]:
print(train_gen.class_indices)

{'biological': 0, 'cardboard': 1, 'glass': 2, 'metal': 3, 'paper': 4, 'plastic': 5, 'trash': 6}


In [13]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)

output = layers.Dense(7, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [14]:
model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=epochs
)

Epoch 1/5
364/364 ━━━━━━━━━━━━━━━━━━━━ 130s 348ms/step - accuracy: 0.8411 - loss: 0.4525 - val_accuracy: 0.8512 - val_loss: 0.4130
Epoch 2/5
364/364 ━━━━━━━━━━━━━━━━━━━━ 72s 199ms/step - accuracy: 0.9264 - loss: 0.2108 - val_accuracy: 0.8864 - val_loss: 0.3086
Epoch 3/5
364/364 ━━━━━━━━━━━━━━━━━━━━ 69s 190ms/step - accuracy: 0.9582 - loss: 0.1148 - val_accuracy: 0.8740 - val_loss: 0.4015
Epoch 4/5
364/364 ━━━━━━━━━━━━━━━━━━━━ 67s 184ms/step - accuracy: 0.9749 - loss: 0.0780 - val_accuracy: 0.8657 - val_loss: 0.4323
Epoch 5/5
364/364 ━━━━━━━━━━━━━━━━━━━━ 72s 197ms/step - accuracy: 0.9854 - loss: 0.0483 - val_accuracy: 0.8850 - val_loss: 0.3842


In [15]:
model.save("model.h5")

In [1]:
import tensorflow as tf

model = tf.keras.models.load_model("model.h5")
model.save("model.keras")